# Backtesting inputs and performance

This offline notebook constructs immutable replay inputs and calculates performance from a deterministic equity curve. A full run additionally requires canonical Postgres bars and an injected strategy/risk manager.

In [ ]:
from datetime import UTC, datetime, timedelta
from trader.backtest import (
    BacktestSpec, EquityPoint, build_backtest_assumptions,
    _build_performance_summary,
)

start = datetime(2026, 1, 1, 9, 0, tzinfo=UTC)
spec = BacktestSpec(start=start, end=start + timedelta(hours=2), timeframe="1Hour")
assumptions = build_backtest_assumptions({
    "fees": {"fixed_per_order": 0.25},
    "slippage": {"bps": 2.0},
})
assert spec.start <= spec.end
assert assumptions.fees.fixed_per_order == 0.25
assert assumptions.slippage.bps == 2.0

In [ ]:
from math import isclose

curve = (
    EquityPoint(ts=start, equity=100_000.0),
    EquityPoint(ts=start + timedelta(hours=1), equity=99_000.0),
    EquityPoint(ts=start + timedelta(hours=2), equity=102_000.0),
)
summary = _build_performance_summary(curve, spec.timeframe, exposure_samples=())
assert isclose(summary.total_return, 0.02)
assert isclose(summary.max_drawdown, 0.01)
assert summary.end_equity == 102_000.0

The result is meaningful only with its replay scope, data identity, implementation/specification versions, assumptions, warnings, trade ledger, and benchmark. Continue in `backtesting.md` for the Postgres-backed lifecycle.